In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Simulação exata com as primitivas do Qiskit SDK

<details>
<summary><b>Versões dos pacotes</b></summary>

O código nesta página foi desenvolvido usando os seguintes requisitos.
Recomendamos usar essas versões ou versões mais recentes.

```
qiskit[all]~=2.3.0
```
</details>
As primitivas de referência do Qiskit SDK realizam simulações locais de vetor de estado. Essas simulações não suportam
a modelagem de ruído de dispositivos, mas são úteis para prototipagem rápida de algoritmos antes de explorar técnicas de simulação mais avançadas ([usando o Qiskit Aer](./simulate-stabilizer-circuits)) ou executar em dispositivos reais ([primitivas do Qiskit Runtime](primitives)).

A primitiva Estimator pode calcular valores esperados de circuitos, e a primitiva Sampler pode amostrar
das distribuições de saída de circuitos.

As seções a seguir mostram como usar as primitivas de referência para executar seu fluxo de trabalho localmente.

## Use o Estimator de referência
A implementação de referência do `EstimatorV2` em `qiskit.primitives` que roda em simuladores locais de vetor de estado
é a classe [`StatevectorEstimator`](../api/qiskit/qiskit.primitives.StatevectorEstimator). Ela aceita circuitos, observáveis e parâmetros como entradas e retorna os valores esperados calculados localmente.

O código a seguir prepara as entradas que serão usadas nos exemplos posteriores. O tipo de entrada esperado para os
observáveis é [`qiskit.quantum_info.SparsePauliOp`](../api/qiskit/qiskit.quantum_info.SparsePauliOp). Note que
o circuito do exemplo é parametrizado, mas você também pode executar o Estimator em circuitos não parametrizados.

> **Note:** Qualquer circuito passado para um Estimator **não** deve incluir nenhuma **medição**.

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter

# circuit for which you want to obtain the expected value
circuit = QuantumCircuit(2)
circuit.ry(Parameter("theta"), 0)
circuit.h(0)
circuit.cx(0, 1)
circuit.draw("mpl", style="iqp")

<Image src="../docs/images/guides/simulate-with-qiskit-sdk-primitives/extracted-outputs/5b41a52d-8f15-4ce4-b3f6-effd91946d9c-0.svg" alt="Output of the previous code cell" />

In [2]:
from qiskit.quantum_info import SparsePauliOp
import numpy as np

# observable(s) whose expected values you want to compute

observable = SparsePauliOp(["II", "XX", "YY", "ZZ"], coeffs=[1, 1, -1, 1])

# value(s) for the circuit parameter(s)
parameter_values = [[0], [np.pi / 6], [np.pi / 2]]

> **Tip:** O fluxo de trabalho das primitivas do Qiskit Runtime exige que circuitos e observáveis sejam transformados para usar apenas instruções suportadas pelo QPU (denominados circuitos e observáveis de *arquitetura de conjunto de instruções (ISA)*). As primitivas de referência ainda aceitam instruções abstratas, pois dependem de simulações locais de vetor de estado, mas transpilar o circuito ainda pode ser benéfico em termos de otimização.
> 
>   ```python
>   # Generate a pass manager without providing a backend
>   from qiskit.transpiler import generate_preset_pass_manager
> 
>   pm = generate_preset_pass_manager(optimization_level=1)
>   isa_circuit = pm.run(circuit)
>   isa_observable = observable.apply_layout(isa_circuit.layout)
>   ```

### Inicializar o Estimator
Instancie um [`qiskit.primitives.StatevectorEstimator`](../api/qiskit/qiskit.primitives.StatevectorEstimator).

In [3]:
from qiskit.primitives import StatevectorEstimator

estimator = StatevectorEstimator()

### Executar e obter resultados
Este exemplo usa apenas um circuito (do tipo [`QuantumCircuit`](../api/qiskit/qiskit.circuit.QuantumCircuit)) e um
observável.

Execute a estimativa chamando o método [`StatevectorEstimator.run`](../api/qiskit/qiskit.primitives.StatevectorEstimator#run), que retorna uma instância de um objeto [`PrimitiveJob`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveJob). Você pode obter os resultados do job (como um objeto [`qiskit.primitives.PrimitiveResult`](../api/qiskit/qiskit.primitives.PrimitiveResult))
com o método [`qiskit.primitives.PrimitiveJob.result`](../api/qiskit/qiskit.primitives.PrimitiveJob#result).

In [4]:
job = estimator.run([(circuit, observable, parameter_values)])
result = job.result()
print(f" > Result class: {type(result)}")

 > Result class: <class 'qiskit.primitives.containers.primitive_result.PrimitiveResult'>


#### Get the expected value from the result

The primitives result outputs an array of [`PubResult`](/docs/api/qiskit/qiskit.primitives.PubResult#pubresult) objects, where each item of the array is a `PubResult` object that contains in its data the array of evaluations corresponding to every circuit-observable combination in the PUB.

To retrieve the expectation values and metadata for the first (and in this case, only) circuit evaluation, we must access the evaluation [`data`](/docs/api/qiskit/qiskit.primitives.PubResult#data) for PUB 0:

In [5]:
print(f" > Expectation value: {result[0].data.evs}")
print(f" > Metadata: {result[0].metadata}")

 > Expectation value: [4.         3.73205081 2.        ]
 > Metadata: {'target_precision': 0.0, 'circuit_metadata': {}}


#### Obter o valor esperado do resultado
Os resultados das primitivas retornam um array de objetos [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult#pubresult), onde cada item do array é um objeto `PubResult` que contém em seus dados o array de avaliações correspondente a cada combinação circuito-observável no PUB.

Para recuperar os valores esperados e os metadados da primeira (e, neste caso, única) avaliação do circuito, precisamos acessar os [`data`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult#data) de avaliação do PUB 0:

In [6]:
# Estimate expectation values for two PUBs, both with 0.05 precision.
precise_job = estimator.run(
    [(circuit, observable, parameter_values)], precision=0.05
)

For a full example, see the [Primitives examples](primitives-examples#estimator-examples) page.

## Use the reference Sampler

The reference implementations of `SamplerV2` in `qiskit.primitives` is the [`StatevectorSampler`](../api/qiskit/qiskit.primitives.StatevectorSampler) class. It takes circuits and parameters as inputs and returns the results from sampling from the output probability distributions as a quasi-probability distribution of output states.

The following code prepares the inputs used in the examples that follow. Note that
these examples run a single parametrized circuit, but you can also run the Sampler
on non-parametrized circuits.

In [7]:
from qiskit import QuantumCircuit

circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure_all()
circuit.draw("mpl", style="iqp")

<Image src="../docs/images/guides/simulate-with-qiskit-sdk-primitives/extracted-outputs/d4c0ac3b-8e5b-4cde-bb26-256324982c2c-0.svg" alt="Output of the previous code cell" />

### Definir opções de execução do Estimator
Por padrão, o Estimator de referência realiza um cálculo exato de vetor de estado baseado na
classe [`quantum_info.Statevector`](../api/qiskit/qiskit.quantum_info.Statevector).
No entanto, isso pode ser modificado para introduzir o efeito da sobrecarga de amostragem (também conhecida como "ruído de shot").

O Estimator aceita um argumento `precision` que expressa as barras de erro que a
implementação da primitiva deve almejar para estimativas de valores esperados. Isso é a sobrecarga de amostragem e é definida exclusivamente no método `.run()`. Isso permite ajustar a opção até o nível do PUB.

In [8]:
from qiskit.primitives import StatevectorSampler

sampler = StatevectorSampler()

Para um exemplo completo, consulte a página de [exemplos de Primitivas](primitives-examples#estimator-examples).

## Use o Sampler de referência
A implementação de referência do `SamplerV2` em `qiskit.primitives` é a classe [`StatevectorSampler`](../api/qiskit/qiskit.primitives.StatevectorSampler). Ela aceita circuitos e parâmetros como entradas e retorna os resultados da amostragem das distribuições de probabilidade de saída como uma distribuição quasi-probabilística dos estados de saída.

O código a seguir prepara as entradas usadas nos exemplos posteriores. Note que
esses exemplos executam um único circuito parametrizado, mas você também pode executar o Sampler
em circuitos não parametrizados.

In [9]:
# execute 1 circuit with Sampler
job = sampler.run([circuit])
pub_result = job.result()[0]
print(f" > Result class: {type(pub_result)}")

 > Result class: <class 'qiskit.primitives.containers.sampler_pub_result.SamplerPubResult'>


![Output of the previous code cell](../docs/images/guides/simulate-with-qiskit-sdk-primitives/extracted-outputs/d4c0ac3b-8e5b-4cde-bb26-256324982c2c-0.svg)

> **Note:** Qualquer circuito quântico passado para um Sampler **deve** incluir medições.

> **Tip:** O fluxo de trabalho das primitivas do Qiskit Runtime exige que os circuitos sejam transformados para usar apenas instruções suportadas pelo QPU (denominados circuitos ISA). As primitivas de referência ainda aceitam instruções abstratas, pois dependem de simulações locais de vetor de estado, mas transpilar o circuito ainda pode ser benéfico em termos de otimização.
> 
>   ```python
>   # Generate a pass manager without providing a backend
>   from qiskit.transpiler import generate_preset_pass_manager
> 
>   pm = generate_preset_pass_manager(optimization_level=1)
>   isa_circuit = pm.run(qc)
>   ```

### Inicializar o `SamplerV2`
Instancie [`qiskit.primitives.StatevectorSampler`](../api/qiskit/qiskit.primitives.StatevectorSampler):

In [10]:
from qiskit.transpiler import generate_preset_pass_manager

# create two circuits
circuit1 = circuit.copy()
circuit2 = circuit.copy()

# transpile circuits
pm = generate_preset_pass_manager(optimization_level=1)
isa_circuit1 = pm.run(circuit1)
isa_circuit2 = pm.run(circuit2)
# execute 2 circuits using Sampler
job = sampler.run([(isa_circuit1), (isa_circuit2)])
pub_result_1 = job.result()[0]
pub_result_2 = job.result()[1]
print(f" > Result class: {type(pub_result)}")

 > Result class: <class 'qiskit.primitives.containers.sampler_pub_result.SamplerPubResult'>


### Executar e obter resultados

In [11]:
# Define quantum circuit with 2 qubits
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure_all()
circuit.draw()

        ┌───┐      ░ ┌─┐   
   q_0: ┤ H ├──■───░─┤M├───
        └───┘┌─┴─┐ ░ └╥┘┌─┐
   q_1: ─────┤ X ├─░──╫─┤M├
             └───┘ ░  ║ └╥┘
meas: 2/══════════════╩══╩═
                      0  1 

In [12]:
# Transpile circuit
pm = generate_preset_pass_manager(optimization_level=1)
isa_circuit = pm.run(circuit)
# Run using sampler
result = sampler.run([circuit]).result()
# Access result data for PUB 0
data_pub = result[0].data
# Access bitstring for the classical register "meas"
bitstrings = data_pub.meas.get_bitstrings()
print(f"The number of bitstrings is: {len(bitstrings)}")
# Get counts for the classical register "meas"
counts = data_pub.meas.get_counts()
print(f"The counts are: {counts}")

The number of bitstrings is: 1024
The counts are: {'11': 515, '00': 509}


As primitivas aceitam múltiplos PUBs como entradas, e cada PUB recebe seu próprio resultado. Portanto, você pode executar circuitos diferentes com várias combinações de parâmetros/observáveis e recuperar os resultados dos PUBs:

In [13]:
# Sample two circuits at 128 shots each.
sampler.run([isa_circuit1, isa_circuit2], shots=128)
# Sample two circuits at different amounts of shots. The "None"s are necessary
# as placeholders
# for the lack of parameter values in this example.
sampler.run([(isa_circuit1, None, 123), (isa_circuit2, None, 456)])

For a full example, see the [Primitives examples](./primitives-examples#sampler-examples) page.
## Next steps

<Admonition type="tip" title="Recommendations">
  - For higher-performance simulation that can handle larger circuits, or to incorporate noise models into your simulation, see [Exact and noisy simulation with Qiskit Aer primitives](simulate-with-qiskit-aer).
  - To learn how to use Quantum Composer for simulation, see the [IBM Quantum Composer](/docs/guides/composer) guide.
  - Read the [Qiskit Estimator API](/docs/api/qiskit/1.4/qiskit.primitives.Estimator) reference.
  - Read the [Qiskit Sampler API](/docs/api/qiskit/1.4/qiskit.primitives.Sampler) reference.
  - Read [Migrate to V2 primitives](/docs/guides/v2-primitives).
</Admonition>